# HERMES Qwen2.5-1.5B — QLoRA Training
Select T4 GPU in Runtime settings. Free GPU availability is not guaranteed. No paid subscription required. Synthetic examples only.

In [ ]:
!pip -q install -U transformers peft accelerate bitsandbytes datasets trl safetensors

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU unavailable: stop here; do not switch to paid service automatically.'
print(torch.cuda.get_device_name(0))

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer
model_id='Qwen/Qwen2.5-1.5B-Instruct'
examples=[('ค้นหางานค้างใน Greenhouse','FIND'),('รับคำขอสร้างภาพ','INTAKE'),('ส่งผลลัพธ์กลับ HALL','RETURN'),('ตรวจสถานะงาน','STATUS'),('ตรวจผลทดสอบ','VERIFY'),('จัดทำรายงาน','REPORT'),('ค้นหาเครื่องมือที่รองรับ','DISCOVER'),('เริ่มงานที่ได้รับมอบหมาย','EXECUTE')]
rows=[{'text':f'<|im_start|>system\nClassify the user request into one command. Reply with command name only.<|im_end|>\n<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{label}<|im_end|>'} for q,label in examples]*8
dataset=Dataset.from_list(rows)
tokenizer=AutoTokenizer.from_pretrained(model_id)
model=AutoModelForCausalLM.from_pretrained(model_id,quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16),device_map='auto')

In [ ]:
args=TrainingArguments(output_dir='/content/hermes-training',per_device_train_batch_size=1,gradient_accumulation_steps=4,num_train_epochs=1,learning_rate=2e-4,logging_steps=2,save_strategy='no',report_to='none',fp16=True)
trainer=SFTTrainer(model=model,train_dataset=dataset,args=args,peft_config=LoraConfig(r=8,lora_alpha=16,target_modules=['q_proj','k_proj','v_proj','o_proj'],task_type='CAUSAL_LM'),processing_class=tokenizer,dataset_text_field='text',max_seq_length=256)
trainer.train()
trainer.model.save_pretrained('/content/hermes-lora')
tokenizer.save_pretrained('/content/hermes-lora')

## Save output
Use the Colab Files panel to download `/content/hermes-lora` as an archive or copy it to your own Drive. Training and inference accuracy have NOT been verified by this notebook. If TRL API has changed, inspect the error rather than silently changing the recipe.